In [ ]:
###############  PHASE 2  ################
#########STEP 6- Spatial flood risk mapping 
#Lagos abd Durban - DEM + Built surface analysis

##Purpose
#This notebook develops spatial flood risk maps by integrating digital elevation models, terrain slope 
#and built surface density into a composite flood risk index for Lagos and Durban.

## Research Context
#Flood vulnerability varies spatially because of differences in terrain, 
#urban density and land characteristics. Spatial modelling complements the 
#statistical analyses by identifying areas where flood risk is concentrated.

##Input Data
#Digital Elevation Models (DEM)
#Built Surface raster datasets

## Methods
#Raster preprocessing
#Slope calculation
#Percentile normalisation
#Weighted overlay analysis
#Composite flood risk scoring

## Outputs
#Elevation maps
#Slope maps
#Built surface density maps
#Composite flood risk maps




##load packages
import numpy as np
import rasterio
import matplotlib.pyplot as plt
import pandas as pd
import os
from rasterio.warp import reproject, Resampling


In [ ]:
#File paths
paths = {
    "lagos": {
        "dem": r"C:\Users\drsih\Desktop\profile\flooding_project\data\raw\topography\dem_gedtm30_lagos.tif",
        "built": r"C:\Users\drsih\Desktop\profile\flooding_project\data\raw\lulc\ghs_built_surface_2020_lagos.tif",
    },
    "durban": {
        "dem": r"C:\Users\drsih\Desktop\profile\flooding_project\data\raw\topography\dem_gedtm30_durban.tif",
        "built": r"C:\Users\drsih\Desktop\profile\flooding_project\data\raw\lulc\ghs_built_surface_2020_durban.tif",
    }
}

OUTPUT_DIR = r"C:\Users\drsih\Desktop\profile\flooding_project\outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_DIR
paths

In [ ]:
###Helper function
def load_raster(path):

    with rasterio.open(path) as src:

        data = src.read(1).astype(np.float32)

        data[data == src.nodata] = np.nan

        return (
            data,
            src.meta.copy(),
            src.transform,
            src.crs
        )

In [ ]:

def percentile_normalise(arr):

    lo = np.nanpercentile(arr, 2)
    hi = np.nanpercentile(arr, 98)

    return np.clip((arr - lo) / (hi - lo), 0, 1)

In [ ]:

def compute_slope(dem, res_x, res_y):

    dz_dx = np.gradient(
        dem,
        float(res_x),
        axis=1
    )

    dz_dy = np.gradient(
        dem,
        float(res_y),
        axis=0
    )

    slope_rad = np.arctan(
        np.sqrt(dz_dx**2 + dz_dy**2)
    )

    return np.degrees(slope_rad)

In [ ]:

def align_built_to_dem(dem_path, built_path):

    with rasterio.open(dem_path) as dem_src:

        dem = dem_src.read(1).astype(np.float32)

        dem[dem == dem_src.nodata] = np.nan

        dem_transform = dem_src.transform
        dem_crs = dem_src.crs
        dem_shape = dem.shape

    with rasterio.open(built_path) as built_src:

        built = built_src.read(1).astype(np.float32)

        built_aligned = np.empty(
            dem_shape,
            dtype=np.float32
        )

        reproject(
            source=built,
            destination=built_aligned,
            src_transform=built_src.transform,
            src_crs=built_src.crs,
            dst_transform=dem_transform,
            dst_crs=dem_crs,
            resampling=Resampling.bilinear
        )

    return (
        dem,
        built_aligned,
        dem_transform,
        dem_crs
    )

In [ ]:

def classify_risk(risk):

    classes = np.full(
        risk.shape,
        np.nan
    )

    classes[risk < 0.3] = 1

    classes[
        (risk >= 0.3) &
        (risk < 0.6)
    ] = 2

    classes[risk >= 0.6] = 3

    return classes

In [ ]:

def compute_flood_risk(
    dem,
    built,
    transform,
    city
):

    res_x = abs(transform.a)
    res_y = abs(transform.e)

    slope = compute_slope(
        dem,
        res_x,
        res_y
    )

    elev_risk = 1 - percentile_normalise(dem)

    flat_risk = 1 - percentile_normalise(slope)

    steep_risk = percentile_normalise(slope)

    if city == "lagos":

        slope_risk = (
            0.7 * flat_risk +
            0.3 * steep_risk
        )

    else:

        slope_risk = (
            0.4 * flat_risk +
            0.6 * steep_risk
        )

    built_risk = percentile_normalise(built)

    risk = (
        0.4 * elev_risk +
        0.3 * slope_risk +
        0.3 * built_risk
    )

    risk[np.isnan(dem)] = np.nan

    return risk, slope

In [ ]:
#main analysis loop
results = {}

for city in ["lagos", "durban"]:

    print(f"\nProcessing {city}")

    dem, built, transform, crs = align_built_to_dem(
        paths[city]["dem"],
        paths[city]["built"]
    )

    risk, slope = compute_flood_risk(
        dem,
        built,
        transform,
        city
    )

    risk_class = classify_risk(risk)

    results[city] = {
    "dem": dem,
    "built": built,
    "risk": risk,
    "slope": slope,
    "risk_class": risk_class,
    "transform": transform,
    "crs": crs
}

   

print("Finished")

In [ ]:
 print(
        f"Mean risk score: {np.nanmean(risk):.3f}"
    )

In [ ]:
#Visualisation - 4-panel map per city

import os

for city in ["lagos", "durban"]:

    r = results[city]

    fig, axes = plt.subplots(
        2,
        2,
        figsize=(14, 12)
    )

    fig.suptitle(
        f"{city.upper()} Flood Risk Assessment",
        fontsize=16,
        fontweight="bold"
    )

    layers = [

        (
            r["dem"],
            "Elevation (m)",
            "terrain"
        ),

        (
            r["slope"],
            "Slope (degrees)",
            "YlOrRd"
        ),

        (
            r["built"],
            "Built Surface Density",
            "Blues"
        ),

        (
            r["risk"],
            "Composite Flood Risk",
            "RdYlGn_r"
        )

    ]

    for ax, (layer, title, cmap) in zip(
        axes.flatten(),
        layers
    ):

        vmin = np.nanpercentile(layer, 2)
        vmax = np.nanpercentile(layer, 98)

        im = ax.imshow(
            layer,
            cmap=cmap,
            vmin=vmin,
            vmax=vmax
        )

        ax.set_title(
            title,
            fontsize=12
        )

        ax.axis("off")

        plt.colorbar(
            im,
            ax=ax,
            fraction=0.046,
            pad=0.04
        )

    plt.tight_layout()

    save_path = os.path.join(
        OUTPUT_DIR,
        f"{city}_risk_maps.png"
    )

    plt.savefig(
        save_path,
        dpi=300,
        bbox_inches="tight"
    )

    print(f"Saved: {save_path}")

    plt.show()

    plt.close()

In [ ]:
########Comparisob map 

fig, axes = plt.subplots(
    1,
    2,
    figsize=(15, 7)
)

fig.suptitle(
    "Composite Flood Risk: Lagos vs Durban",
    fontsize=16,
    fontweight="bold"
)

for ax, city in zip(
    axes,
    ["lagos", "durban"]
):

    im = ax.imshow(
        results[city]["risk"],
        cmap="RdYlGn_r",
        vmin=0,
        vmax=1
    )

    ax.set_title(
        city.capitalize(),
        fontsize=13
    )

    ax.axis("off")

#Leave room on the right for the colour bar
fig.subplots_adjust(right=0.88)

#Create a separate axis for the colour bar
cbar_ax = fig.add_axes([0.90, 0.15, 0.02, 0.70])
#                 left  bottom width height

cbar = fig.colorbar(
    im,
    cax=cbar_ax
)

cbar.set_label(
    "Flood Risk Score (0 = Low, 1 = High)",
    fontsize=11
)

comparison_path = os.path.join(
    OUTPUT_DIR,
    "lagos_durban_risk_comparison.png"
)

plt.savefig(
    comparison_path,
    dpi=300,
    bbox_inches="tight"
)

print(f"Saved: {comparison_path}")

plt.show()

plt.close()

In [ ]:
#Risk distribution Summary table
print("\n")
print("=" * 60)
print("FLOOD RISK ZONE DISTRIBUTION")
print("=" * 60)

summary_rows = []

for city in ["lagos", "durban"]:

    risk = results[city]["risk"]

    rc = results[city]["risk_class"]

    valid = np.isfinite(rc)

    total = valid.sum()

    low = (rc == 1).sum()

    medium = (rc == 2).sum()

    high = (rc == 3).sum()

    mean_risk = np.nanmean(risk)

    summary_rows.append({

        "City": city.capitalize(),

        "Mean_Risk":
            round(mean_risk, 3),

        "Low_Risk_%":
            round(100 * low / total, 1),

        "Medium_Risk_%":
            round(100 * medium / total, 1),

        "High_Risk_%":
            round(100 * high / total, 1)

    })

    print(f"\n{city.upper()}")

    print(
        f"Mean Risk Score : {mean_risk:.3f}"
    )

    print(
        f"Low Risk Area   : {100 * low / total:.1f}%"
    )

    print(
        f"Medium Risk Area: {100 * medium / total:.1f}%"
    )

    print(
        f"High Risk Area  : {100 * high / total:.1f}%"
    )

summary_table = pd.DataFrame(
    summary_rows
)

print("\n")
print(summary_table)

summary_table.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "step6_risk_summary.csv"
    ),
    index=False
)

print(
    "\nSaved: step6_risk_summary.csv"
)

In [ ]:
##Summary

##Main Outcomes
#Generated composite flood risk maps for Lagos and Durban.
#Identified spatial hotspots of elevated flood risk.
#Compared contrasting terrain and urban development patterns between the two cities.

##Limitations
#The flood risk index represents relative susceptibility rather than simulated flood depth or inundation extent.

##Next Notebook
#->**07_scenario_modelling_and_cba.ipynb**
#The final notebook evaluates mitigation scenarios and estimates their potential economic benefits through cost-benefit analysis.